# Thulla DMC — Colab training (Rust env)

Minimal DouZero-style self-play for a decent 4-player Thulla bot.

Learns **card plays** and **ASK/PASS** on the take phase (victims always give).

Obs include known holdings (thulla/take) and the **free unknown** card pool; heads-up uses deduced opponent hands.

**Setup:** Prefer **T4 GPU** when available. This notebook also runs on **CPU** (defaults below).

- Learner on GPU or CPU; self-play actors on **CPU** (default **3×8** envs, batched inference)
- Latest checkpoint every ~3 min → `model.tar`
- **Eval vs random** every 15 min (50 games)
- **Eval vs heuristic** every 30 min (50 games) → saves `model_best.tar` when P(not last) improves
- Eval summaries append to **`eval_log.csv`** in the same Drive folder

**Note:** If you changed obs/encoding recently, start a **fresh** checkpoint folder or delete old `model.tar`.


## 1. Install deps

In [1]:
%pip install -q "numpy>=1.24" "torch>=2.0"

## 2. Get thulla-ai code

Either clone your repo, or upload a zip of `thulla-ai` to Drive and set `REPO_DIR` below.

In [2]:
import os
import sys

import torch

# --- edit these ---
REPO_URL = "https://github.com/ahmedpervaiz31/thulla-ai.git"
REPO_DIR = "/content/thulla-ai"
DRIVE_CKPT = "/content/drive/MyDrive/thulla_dmc_ckpts"
# ------------------

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CKPT, exist_ok=True)

if REPO_URL and not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif not os.path.isdir(REPO_DIR):
    raise SystemExit(
        f"Missing {REPO_DIR}. Set REPO_URL or upload thulla-ai there (must contain thulla/ and thulla_dmc/)."
    )

sys.path.insert(0, REPO_DIR)

# if not torch.cuda.is_available():
#     raise SystemExit(
#         "No CUDA GPU. Runtime → Change runtime type → Hardware accelerator → T4 GPU, then re-run."
#     )

# print("REPO_DIR:", REPO_DIR)
# print("checkpoints:", DRIVE_CKPT)
# print("GPU:", torch.cuda.get_device_name(0))
# print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

print("REPO_DIR:", REPO_DIR)
print("checkpoints:", DRIVE_CKPT)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("Running in CPU mode (No GPU hardware accelerator).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
REPO_DIR: /content/thulla-ai
checkpoints: /content/drive/MyDrive/thulla_dmc_ckpts
Running in CPU mode (No GPU hardware accelerator).


## 3. Train — reward reshape run (`thulla_dmc_r2`)

**New finish rewards:** `+2.0, +1.5, +1.0, −1.0` (was `+1.2, +1.1, +1.0, −1.0`).

- Saves under **`thulla_dmc_r2`** (old `thulla_dmc` / `model_best` untouched).
- First start: **`--init_from`** old `model_best.tar` (weights only, episodes=0, fresh optimizer).
- Later: auto-resume `thulla_dmc_r2/model.tar` if present.
- CPU preset: 3 actors × 32 envs. Evals on (random 15 min / heuristic 30 min) + **heuristic pick agree** diagnostic.
- Rebuild Rust after pull (`maturin develop --release`) so native rewards match Python.


In [ ]:
import os
from thulla_dmc.arguments import parse_args
from thulla_dmc.train import train

XPID = "thulla_dmc_r2"  # new reward run; keep thulla_dmc/ as baseline
ckpt_dir = os.path.join(DRIVE_CKPT, XPID)
old_best = os.path.join(DRIVE_CKPT, "thulla_dmc", "model_best.tar")
old_main = os.path.join(DRIVE_CKPT, "thulla_dmc", "model.tar")
init_src = old_best if os.path.isfile(old_best) else old_main
has_ckpt = os.path.isfile(os.path.join(ckpt_dir, "model.tar"))

# Rewards +2/+1.5/+1/-1 in code (Python + Rust). CPU: 3x32.
argv = [
    "--savedir", DRIVE_CKPT,
    "--xpid", XPID,
    "--training_device", "cpu",
    "--num_actors", "3",
    "--actor_envs", "32",
    "--batch_size", "512",
    "--save_interval", "3",
    "--eval_random_minutes", "15",
    "--eval_heuristic_minutes", "30",
    "--eval_games", "50",
    "--eval_seed", "10000",
    "--total_episodes", "100000",
    "--exp_epsilon", "0.05",
    "--log_interval", "25",
]

if has_ckpt:
    argv.append("--load_model")
    print("Resuming r2 from:", ckpt_dir)
elif os.path.isfile(init_src):
    argv.extend(["--init_from", init_src])
    print("New reward run ->", ckpt_dir)
    print("Init weights from:", init_src)
else:
    print("Starting fresh (no old ckpt) ->", ckpt_dir)

flags = parse_args(argv)
train(flags)

[INFO 2026-09-23 08:55:31,858] Training on cpu | actors=6 | batch=512 | eval_seed=10000


Resuming from checkpoint: /content/drive/MyDrive/thulla_dmc_ckpts/thulla_dmc


[INFO 2026-09-23 08:55:36,315] Resumed from /content/drive/MyDrive/thulla_dmc_ckpts/thulla_dmc/model.tar (episodes=217556)
[INFO 2026-09-23 08:55:54,447] episodes=217581 loss=0.4296 mean_target=0.189 buffer=305 P(not last) random=0.940 heuristic=0.420 best_h=0.460
[INFO 2026-09-23 08:56:07,182] episodes=217606 loss=0.5574 mean_target=0.310 buffer=58 P(not last) random=0.940 heuristic=0.420 best_h=0.460
[INFO 2026-09-23 08:56:15,917] episodes=217631 loss=0.3723 mean_target=0.380 buffer=109 P(not last) random=0.940 heuristic=0.420 best_h=0.460
[INFO 2026-09-23 08:56:24,125] episodes=217656 loss=0.8204 mean_target=0.444 buffer=206 P(not last) random=0.940 heuristic=0.420 best_h=0.460
[INFO 2026-09-23 08:56:32,053] episodes=217681 loss=0.5623 mean_target=0.385 buffer=209 P(not last) random=0.940 heuristic=0.420 best_h=0.460
[INFO 2026-09-23 08:56:37,804] episodes=217706 loss=0.7601 mean_target=0.414 buffer=353 P(not last) random=0.940 heuristic=0.420 best_h=0.460
[INFO 2026-09-23 08:56:44,

In [3]:
from thulla_dmc.rust_env import make_env, RustThullaEnv, rust_available
print("flag:", rust_available())
env = make_env(prefer_rust=True)
print("env type:", type(env).__name__)
print("is Rust:", isinstance(env, RustThullaEnv))

flag: True
env type: RustThullaEnv
is Rust: True


In [ ]:
import os
from thulla_dmc.evaluate import evaluate

# Prefer new reward-run best; fall back to baseline thulla_dmc
for xpid in ("thulla_dmc_r2", "thulla_dmc"):
    best = f"{DRIVE_CKPT}/{xpid}/model_best.tar"
    latest = f"{DRIVE_CKPT}/{xpid}/model.tar"
    if os.path.exists(best) or os.path.exists(latest):
        ckpt = best if os.path.exists(best) else latest
        break
else:
    raise SystemExit("No checkpoint under thulla_dmc_r2 or thulla_dmc")

print("Evaluating:", ckpt)

print("\n--- vs random ---")
evaluate(ckpt, num_games=200, device="cpu", opponent="random")

print("\n--- vs heuristic ---")
evaluate(ckpt, num_games=100, device="cpu", opponent="heuristic")

In [5]:
!git -C /content/thulla-ai pull

remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 60 (delta 20), reused 60 (delta 20), pack-reused 0 (from 0)
Unpacking objects: 100% (60/60), 82.96 KiB | 2.68 MiB/s, done.
From https://github.com/ahmedpervaiz31/thulla-ai
   8a7fa62..049f9b4  main       -> origin/main
Updating 8a7fa62..049f9b4
Fast-forward
 README.md                                          |  10 +-
 checkpoints/.gitignore                             |   3 +
 checkpoints/README.md                              |   3 +
 native/thulla_rust/Cargo.lock                      | 347 +++++++++++++++++++++
 native/thulla_rust/Cargo.toml                      |  15 +
 native/thulla_rust/README.md                       |   7 +
 native/thulla_rust/pyproject.toml                  |  13 +
 native/thulla_rust/src/card.rs                     | 116 +++++++
 native/thulla_rust/src/encode.rs                   | 167 ++++++++++
 native/thulla_rus

In [9]:
from thulla_dmc.rust_env import rust_available
print(rust_available())  # must be True

True


In [6]:
%pip install -q maturin
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += ":" + os.path.expanduser("~/.cargo/bin")
%cd /content/thulla-ai/native/thulla_rust
!python -m maturin develop --release

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 59.7 MB/s eta 0:00:00
info: downloading installer
info: profile set to default
info: default host tuple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-09-03 for version 1.98.1 (48a229cea 2026-09-01)
info: downloading 6 components
      rustfmt installed                        2.37 MiB                         info: default toolchain set to stable-x86_64-unknown-linux-gnu

  stable-x86_64-unknown-linux-gnu installed - rustc 1.98.1 (48a229cea 2026-09-01)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdks

In [7]:
import os, sys
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# if you aren't already in the crate dir:
%cd /content/thulla-ai/native/thulla_rust

!python -m maturin build --release -o /content/thulla_rust_wheels
!pip install --force-reinstall --no-deps /content/thulla_rust_wheels/*.whl

# verify (must restart runtime OR at least re-import cleanly)
import importlib
if "thulla_rust" in sys.modules:
    del sys.modules["thulla_rust"]
import thulla_rust
from thulla_dmc.rust_env import rust_available
print("thulla_rust OK:", thulla_rust)
print("rust_available:", rust_available())

/content/thulla-ai/native/thulla_rust
    Updating crates.io index
  Downloaded autocfg v1.5.1
  Downloaded cfg-if v1.0.5
  Downloaded getrandom v0.2.17
  Downloaded heck v0.5.0
  Downloaded indoc v2.0.7
  Downloaded rawpointer v0.2.1
  Downloaded rustc-hash v1.1.0
  Downloaded pyo3-macros v0.22.6
  Downloaded rand_core v0.6.4
  Downloaded portable-atomic-util v0.2.8
  Downloaded rand_chacha v0.3.1
  Downloaded unindent v0.2.4
  Downloaded memoffset v0.9.1
  Downloaded ppv-lite86 v0.2.21
  Downloaded pyo3-build-config v0.22.6
  Downloaded target-lexicon v0.12.16
  Downloaded wasi v0.11.1+wasi-snapshot-preview1
  Downloaded num-integer v0.1.47
  Downloaded quote v1.0.47
  Downloaded rustversion v1.0.23
  Downloaded num-complex v0.4.6
  Downloaded once_cell v1.21.4
  Downloaded matrixmultiply v0.3.11
  Downloaded proc-macro2 v1.0.107
  Downloaded unicode-ident v1.0.26
  Downloaded num-traits v0.2.19
  Downloaded pyo3-ffi v0.22.6
  Downloaded pyo3-macros-backend v0.22.6
  Downloaded numpy